# Chess.com Data Collection

This notebook collects public chess data from the Chess.com PubAPI.

The goal of this project is to build a personal chess analytics pipeline for:
- performance analysis
- rating progression
- opening analysis
- behavioral analytics
- visualization projects

Data source:
https://api.chess.com/pub/

## Imports

In [1]:
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime

## User configuration

I am using my own profile here, but you can replace the value of the USERNAME variable with the player you prefer.

In [2]:
USERNAME = "elmurie"

## Project paths

In [3]:
PROJECT_ROOT = Path().resolve().parent

RAW_DATA_DIR = PROJECT_ROOT / "dataset" / "raw" / USERNAME

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_DIR

WindowsPath('C:/Users/pc01a/anaconda_projects/chess/dataset/raw/elmurie')

# API helper

Chess.com requires a valid User-Agent in order to avoid 403/rate limiting.

In [4]:
headers = {
    "User-Agent": "chess-analytics-project"
}

BASE_URL = f"https://api.chess.com/pub/player/{USERNAME}"


def get_json(url):

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Request failed: {url}")
        print(response.status_code)
        return None

    return response.json()

## Download player profile

In [5]:
profile = get_json(BASE_URL)

profile_df = pd.json_normalize(profile)

profile_df.T.head(20)

,0
avatar,https://images.chesscomfiles.com/uploads/v1/us...
player_id,96016480
@id,https://api.chess.com/pub/player/elmurie
url,https://www.chess.com/member/elmurie
username,elmurie
followers,30
country,https://api.chess.com/pub/country/IT
last_online,1778150484
joined,1605008407
status,basic


In [6]:
profile_df.to_csv(
    RAW_DATA_DIR / "profile.csv",
    index=False
)

## Download player stats

In [7]:
stats = get_json(f"{BASE_URL}/stats")

stats_df = pd.json_normalize(stats)

stats_df.T.head(30)

,0
chess_daily.last.rating,712
chess_daily.last.date,1773380333
chess_daily.last.rd,129
chess_daily.best.rating,1041
chess_daily.best.date,1607205002
chess_daily.best.game,https://www.chess.com/game/daily/624828741
chess_daily.record.win,34
chess_daily.record.loss,39
chess_daily.record.draw,0
chess_daily.record.time_per_move,28055


In [8]:
stats_df.to_csv(
    RAW_DATA_DIR / "stats.csv",
    index=False
)

## Game Archives

Chess.com stores games in monthly archives.

The API first returns a list of archive URLs,
then each archive contains the games played during that month.

In [9]:
archives_data = get_json(
    f"{BASE_URL}/games/archives"
)

archives = archives_data["archives"]

len(archives)

67

## Download all games

In [10]:
all_games = []

for archive in archives:

    data = get_json(archive)

    if not data:
        continue

    for game in data.get("games", []):

        all_games.append({

            "date": (
                datetime.fromtimestamp(
                    game.get("end_time")
                ).strftime("%Y-%m-%d %H:%M:%S")
                if game.get("end_time")
                else None
            ),

            "url": game.get("url"),

            "time_class": game.get("time_class"),
            "time_control": game.get("time_control"),

            "white": game.get("white", {}).get("username"),
            "black": game.get("black", {}).get("username"),

            "white_rating": game.get("white", {}).get("rating"),
            "black_rating": game.get("black", {}).get("rating"),

            "white_result": game.get("white", {}).get("result"),
            "black_result": game.get("black", {}).get("result"),

            "eco": game.get("eco"),

            "pgn": game.get("pgn")
        })

## Create Dataframe

In [11]:
games_df = pd.DataFrame(all_games)

games_df.head()

,date,url,time_class,time_control,white,black,white_rating,black_rating,white_result,black_result,eco,pgn
0,2020-11-10 15:30:24,https://www.chess.com/game/live/5719328865,rapid,600,elmurie,TheMrNoName,213,412,resigned,win,https://www.chess.com/openings/Polish-Opening-...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."
1,2020-11-10 15:46:19,https://www.chess.com/game/live/5719418873,rapid,600,smackersmashbot,elmurie,216,345,resigned,win,https://www.chess.com/openings/Kings-Pawn-Open...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."
2,2020-11-10 16:09:31,https://www.chess.com/game/live/5719482469,rapid,600,amkh98,elmurie,371,256,win,checkmated,https://www.chess.com/openings/Kings-Pawn-Open...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."
3,2020-11-10 17:39:54,https://www.chess.com/game/live/5720005454,rapid,600,elmurie,callumfindlay4,193,330,checkmated,win,https://www.chess.com/openings/Vienna-Game-Max...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."
4,2020-11-10 17:51:42,https://www.chess.com/game/live/5720050101,rapid,600,callumfindlay4,elmurie,291,277,resigned,win,https://www.chess.com/openings/Caro-Kann-Defen...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."


## Dataset overview

The dataset looks pretty good!

In [12]:
games_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21005 entries, 0 to 21004
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   date          21005 non-null  object
 1   url           21005 non-null  object
 2   time_class    21005 non-null  object
 3   time_control  21005 non-null  object
 4   white         21005 non-null  object
 5   black         21005 non-null  object
 6   white_rating  21005 non-null  int64 
 7   black_rating  21005 non-null  int64 
 8   white_result  21005 non-null  object
 9   black_result  21005 non-null  object
 10  eco           21005 non-null  object
 11  pgn           21005 non-null  object
dtypes: int64(2), object(10)
memory usage: 1.9+ MB


In [13]:
games_df.describe(include="all")

,date,url,time_class,time_control,white,black,white_rating,black_rating,white_result,black_result,eco,pgn
count,21005,21005,21005,21005,21005,21005,21005.000000,21005.000000,21005,21005,21005,21005
unique,21005,21005,4,11,10349,10324,NaN,NaN,10,10,1450,21005
top,2020-11-10 15:30:24,https://www.chess.com/game/live/5719328865,blitz,180,elmurie,elmurie,NaN,NaN,win,win,https://www.chess.com/openings/Vienna-Game,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat..."
freq,1,1,19823,15800,10487,10518,NaN,NaN,10661,9676,951,1
mean,NaN,NaN,NaN,NaN,NaN,NaN,628.524637,627.565913,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,120.292341,119.647721,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,100.000000,100.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,561.000000,560.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,649.000000,649.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,711.000000,710.000000,NaN,NaN,NaN,NaN


In [14]:
games_df["time_class"].value_counts()

time_class
blitz     19823
rapid       653
bullet      455
daily        74
Name: count, dtype: int64

## Save dataset

In [15]:
games_df.to_csv(
    RAW_DATA_DIR / "games.csv",
    index=False
)

print("Dataset saved.")

Dataset saved.


# Next Steps

The next notebook will focus on:
- cleaning timestamps
- extracting player-side information
- parsing openings
- handling missing values
- feature engineering